# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}`. Через `get(c, 0)` получаем прошлое количество или 0, затем добавляем 1.
2. `[0, 1, 1]`. Функция меняет сам список, поэтому каждый вызов добавляет в него 1.
3. `2`. После `lower()` два названия Алматы совпадают, а множество хранит только уникальные значения.
4. `{'ab': 2, 'abc': 3}`. Ключ `ab` повторяется, поэтому его значение просто записывается заново.
5. Сначала `5`, потом `None`. Функция печатает 5, но ничего не возвращает, поэтому в `x` будет `None`.

In [1]:
# Проверяем ответы
r={}
for c in ["A","B","A"]:
    r[c]=r.get(c,0)+1
print(r)

def f(x):
    x.append(1)
a=[0]
f(a)
f(a)
print(a)

s={c.lower() for c in ["Almaty","ALMATY","Astana"]}
print(len(s))

d={k:len(k) for k in ["ab","abc","ab"]}
print(d)

def g(n):
    if n>0:
        print(n)
x=g(5)
print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [2]:
# orders.csv может лежать рядом с ноутбуком или в корне проекта
import os
csv_path="orders.csv"
if not os.path.exists(csv_path) and os.path.exists("../orders.csv"):
    csv_path="../orders.csv"
if not os.path.exists(csv_path):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:",os.path.exists(csv_path))

orders.csv на месте: True


In [3]:
import csv
with open(csv_path,encoding="utf-8") as f:
    rows=list(csv.DictReader(f))
print(len(rows),"строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [4]:
# База 1
def to_num(s):
    return float(s.replace(" ",""))

def clean_city(s):
    city=s.strip().lower()
    cities={
        "almaty":"Алматы",
        "astana":"Астана",
        "shymkent":"Шымкент",
        "karaganda":"Караганда",
        "aktobe":"Актобе",
        "atyrau":"Атырау"
    }
    return cities.get(city,city.capitalize())

print(to_num("15 990"))
print(clean_city(" ALMATY "))

15990.0
Алматы


In [5]:
# База 2
count_category={}
for row in rows:
    category=row["category"]
    count_category[category]=count_category.get(category,0)+1

sorted_categories=sorted(count_category.items(),key=lambda p:p[1],reverse=True)
for category,count in sorted_categories:
    print(f"{category} | {count}")

Одежда | 1529
Электроника | 1185
Дом | 1105
Красота | 991
Книги | 653


In [6]:
# База 3: средний чек считаем только по оплаченным заказам
revenue_category={}
paid_category={}
for row in rows:
    if row["status"]=="оплачен":
        category=row["category"]
        revenue_category[category]=revenue_category.get(category,0)+to_num(row["amount"])
        paid_category[category]=paid_category.get(category,0)+1

print(f"{'Категория':<15} | {'Выручка':>15} | {'Средний чек':>12}")
for category in count_category:
    revenue=revenue_category.get(category,0)
    count=paid_category.get(category,0)
    average=revenue/count if count else 0
    print(f"{category:<15} | {revenue:>15.2f} | {average:>12.2f}")

Категория       |         Выручка |  Средний чек
Одежда          |     18321080.00 |     13285.77
Красота         |      5507620.00 |      6188.34
Дом             |     10516320.00 |     10654.83
Книги           |      2294100.00 |      3934.99
Электроника     |     65270240.00 |     61286.61


In [7]:
# База 4
count_month={}
for row in rows:
    month=row["order_date"][:7]
    count_month[month]=count_month.get(month,0)+1

for month,count in sorted(count_month.items()):
    print(f"{month} | {count}")

2026-09 | 2782
2026-10 | 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [8]:
# Стандарт 5
revenue_month={}
for row in rows:
    if row["status"]=="оплачен":
        month=row["order_date"][:7]
        revenue_month[month]=revenue_month.get(month,0)+to_num(row["amount"])

for month,revenue in sorted(revenue_month.items()):
    print(f"{month} | {revenue:.2f}")

september=revenue_month["2026-09"]
october=revenue_month["2026-10"]
change=(october-september)/september*100
print(f"Изменение октября к сентябрю: {change:.2f}%")

2026-09 | 57489850.00
2026-10 | 44419510.00
Изменение октября к сентябрю: -22.74%


In [9]:
# Стандарт 6
count_city={}
cancelled_city={}
for row in rows:
    city=clean_city(row["city"])
    count_city[city]=count_city.get(city,0)+1
    if row["status"]=="отменён":
        cancelled_city[city]=cancelled_city.get(city,0)+1

for city,count in count_city.items():
    percent=cancelled_city.get(city,0)/count*100
    print(f"{city} | {percent:.1f}%")

Алматы | 7.9%
Астана | 8.8%
Караганда | 8.9%
Шымкент | 20.4%
Атырау | 7.5%
Актобе | 10.3%


In [10]:
# Стандарт 7
revenue_customer={}
for row in rows:
    if row["status"]=="оплачен":
        customer=row["customer_id"]
        revenue_customer[customer]=revenue_customer.get(customer,0)+to_num(row["amount"])

top_customers=sorted(revenue_customer.items(),key=lambda p:p[1],reverse=True)[:5]
for customer,revenue in top_customers:
    print(f"{customer} | {revenue:.2f}")

best_customer=top_customers[0][0]
print("Заказы клиента на первом месте:")
for row in rows:
    if row["customer_id"]==best_customer:
        print(f"{row['order_date']} | {row['amount']} | {row['status']}")

TEST | 29999970.00
C01627 | 257180.00
C00485 | 215920.00
C01338 | 192080.00
C02122 | 191670.00
Заказы клиента на первом месте:
2026-09-12 11:00 | 9999990.0 | оплачен
2026-09-25 15:00 | 9999990.0 | оплачен
2026-10-03 10:00 | 9999990.0 | оплачен


На первом месте клиент `TEST` с выручкой 29 999 970 ₸. У него три оплаченных заказа по 9 999 990 ₸. Название и одинаковые очень большие суммы похожи на тестовые заказы, которые завышают выручку. В этом расчёте они пока остаются.

In [11]:
# Стандарт 8
customers_city={}
for row in rows:
    city=clean_city(row["city"])
    if city not in customers_city:
        customers_city[city]=set()
    customers_city[city].add(row["customer_id"])

for city,customers in customers_city.items():
    print(f"{city} | {len(customers)}")

Алматы | 1351
Астана | 981
Караганда | 451
Шымкент | 655
Атырау | 138
Актобе | 358


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [12]:
# Челлендж 9
unique_rows=[]
seen=set()
duplicates_month={}
for row in rows:
    row_tuple=tuple(row.values())
    if row_tuple in seen:
        month=row["order_date"][:7]
        duplicates_month[month]=duplicates_month.get(month,0)+1
    else:
        seen.add(row_tuple)
        unique_rows.append(row)

print("Полных дублей:",sum(duplicates_month.values()))
for month,count in sorted(duplicates_month.items()):
    print(f"{month} | {count}")


Полных дублей: 180
2026-09 | 180


In [13]:
# Челлендж 10
revenue_month_clean={}
for row in unique_rows:
    if row["status"]=="оплачен" and row["customer_id"]!=best_customer:
        month=row["order_date"][:7]
        revenue_month_clean[month]=revenue_month_clean.get(month,0)+to_num(row["amount"])

for month,revenue in sorted(revenue_month_clean.items()):
    print(f"{month} | {revenue:.2f}")

september_clean=revenue_month_clean["2026-09"]
october_clean=revenue_month_clean["2026-10"]
change_clean=(october_clean-september_clean)/september_clean*100
print(f"До очистки: {change:.2f}%")
print(f"После очистки: {change_clean:.2f}%")
print(f"Разница: {change_clean-change:.2f} процентного пункта")


2026-09 | 35243410.00
2026-10 | 34419520.00
До очистки: -22.74%
После очистки: -2.34%
Разница: 20.40 процентного пункта


In [14]:
# Челлендж 11: считаем все заказы после удаления дублей
count_customer={}
for row in unique_rows:
    customer=row["customer_id"]
    count_customer[customer]=count_customer.get(customer,0)+1

sorted_customers=sorted(count_customer.items(),key=lambda p:p[1],reverse=True)
customer,count=sorted_customers[0]
print(f"Больше всего заказов у {customer}: {count}")


Больше всего заказов у C01818: 10


После очистки выручка в октябре упала на 2,34%, а не на 22,74%. В сентябре выручка составила 35 243 410 ₸, в октябре — 34 419 520 ₸. В первом расчёте учитывались 180 полных дублей за сентябрь и три заказа клиента TEST по 9 999 990 ₸. Эти записи завышали выручку, особенно в сентябре, поэтому падение казалось намного больше.